# FinSafe 04b — output-budget sensitivity diagnostic
Run after 04 in a session containing its saved predictions. This predeclared diagnostic selects the two smallest IDs among capped outputs in each prompt kind, up to six examples. It uses 768 tokens with otherwise unchanged generation settings. It is a deliberately selected stress subset, not a new safety benchmark or an estimate of ASR. Review full paired outputs before deciding whether all compared conditions need a larger output cap. Final test prompts remain untouched.

In [ ]:
import sys, os, json, subprocess, time, datetime
from pathlib import Path
started=time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START stage04b bootstrap",flush=True)
try:
    project=Path("/kaggle/working/finsafe")
    if not project.exists():
        raise RuntimeError("Restore the project/data or run notebook 01 and 02 first")
    status=subprocess.check_output(["git","-C",str(project),"status","--porcelain","--untracked-files=no"],text=True)
    if status.strip():
        raise RuntimeError("Save tracked-file edits before updating")
    subprocess.run(["git","-C",str(project),"pull","--ff-only"],check=True)
    sys.path.insert(0,str(project/"src"))
    from finsafe.progress import Progress
    progress=Progress(project/"runs/notebook-04b/logs","notebook-04b")
    progress.log(f"DONE bootstrap in {time.monotonic()-started:.1f}s")
except BaseException as error:
    print(f"FAILED bootstrap after {time.monotonic()-started:.1f}s ({type(error).__name__})",flush=True)
    raise


In [ ]:
with progress.stage("01-prerequisites"):
    from kaggle_secrets import UserSecretsClient
    from finsafe.data import read_jsonl
    runtime_env=os.environ.copy()
    runtime_env.update(HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"),
        HF_HOME="/kaggle/working/.cache/huggingface",HF_NAMESPACE="Finsafe-team",
        CUDA_VISIBLE_DEVICES="0",TOKENIZERS_PARALLELISM="false",TRL_ONLY_MODE="1",
        DISABLE_UNSLOTH_FOR_TRL="1",PYTHONUNBUFFERED="1")
    gpu_py=str(project/".venv-gpu/bin/python")
    cli=str(project/".venv-gpu/bin/finsafe")
    revisions=json.loads((project/"runs/environment/model-revisions.json").read_text())
    global_sha=revisions["CohereLabs/tiny-aya-global"]
    filtered=project/"data/processed/tatqa-tokenized"
    curated=filtered/"curated"
    train_rows=read_jsonl(curated/"sft_messages.jsonl")
    if len(train_rows)<40:
        raise RuntimeError("Too few accepted training examples for the 5-step, effective-batch-8 smoke test")
    progress.log(f"Accepted curated training examples: {len(train_rows)}")
    progress.log((filtered/"token_manifest.json").read_text())
    progress.run([cli,"doctor","--gpu"],cwd=project,env=runtime_env)

with progress.stage("01b-stop-configuration-diagnostic"):
    stop_check="import json,sys\nfrom pathlib import Path\nfrom transformers import AutoTokenizer,AutoConfig,GenerationConfig\nmodel='CohereLabs/tiny-aya-global'\nrevision=sys.argv[1]\ntokenizer=AutoTokenizer.from_pretrained(model,revision=revision)\nconfig=AutoConfig.from_pretrained(model,revision=revision)\ntry:\n    generation=GenerationConfig.from_pretrained(model,revision=revision)\n    origin='generation_config.json'\nexcept OSError:\n    generation=GenerationConfig.from_model_config(config)\n    origin='model config fallback'\nreport={'model':model,'revision':revision,'tokenizer_eos_token':tokenizer.eos_token,\n    'tokenizer_eos_id':tokenizer.eos_token_id,'model_eos_id':config.eos_token_id,\n    'generation_eos_id':generation.eos_token_id,'generation_origin':origin,\n    'tokenizer_pad_id':tokenizer.pad_token_id,'special_tokens':tokenizer.special_tokens_map,\n    'special_token_ids':dict(zip(tokenizer.all_special_tokens,tokenizer.all_special_ids)),\n    'chat_template':tokenizer.chat_template}\npath=Path('runs/safety-cap-diagnostic-v1/stop_config.json');path.parent.mkdir(parents=True,exist_ok=True)\npath.write_text(json.dumps(report,indent=2)+'\\n')\nprint(json.dumps({k:v for k,v in report.items() if k not in ['chat_template','special_tokens','special_token_ids']},indent=2))\nprint('Stopping configuration diagnostic saved; no generation settings were changed.')\n"
    progress.run([gpu_py,"-u","-c",stop_check,global_sha],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("02-freeze-cap-diagnostic"):
    from finsafe.data import write_jsonl
    from finsafe.progress import atomic_json
    from finsafe.resume import sha256_file
    source_data=project/"data/processed/safety-pilot-v1/validation.jsonl"
    source_predictions=project/"runs/safety-baseline-pilot-v1/predictions.jsonl"
    questions={r['id']:r for r in read_jsonl(source_data)}
    original=read_jsonl(source_predictions)
    if len(original)!=36 or {p['id'] for p in original}!=set(questions):raise RuntimeError("Run/restore the complete Stage 04 baseline first")
    selected=[]
    for kind in ['harmful_direct','harmful_attack','benign_sensitive']:
        candidates=sorted((p for p in original if questions[p['id']]['kind']==kind and p['output_tokens']>=256),key=lambda p:p['id'])
        selected.extend(p['id'] for p in candidates[:2])
    if not selected:raise RuntimeError("No capped responses to diagnose")
    diagnostic=project/"data/processed/safety-cap-diagnostic-v1/validation.jsonl"
    selection={'source_data_sha256':sha256_file(source_data),'predictions_sha256':sha256_file(source_predictions),'selection':'two smallest capped response IDs per kind','ids':selected,'original_cap':256,'new_cap':768}
    marker=diagnostic.parent/"selection.json"
    if marker.exists() and json.loads(marker.read_text())!=selection:raise RuntimeError("Diagnostic selection changed; use a new version")
    write_jsonl(diagnostic,[questions[i] for i in selected]);atomic_json(marker,selection)
    progress.log(json.dumps(selection,indent=2))


In [ ]:
with progress.stage("03-higher-budget-generation"):
    diagnostic_predictions=project/"runs/safety-cap-diagnostic-v1/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global","--revision",global_sha,"--data",str(diagnostic),"--condition","aligned-cap-diagnostic-768","--out",str(diagnostic_predictions),"--max-new-tokens","768","--load-in-4bit","--resume"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("04-paired-response-review"):
    prior={p['id']:p for p in original}
    enlarged=read_jsonl(diagnostic_predictions)
    paired=project/"runs/safety-cap-diagnostic-v1/paired-review.jsonl"
    pairs=[{'id':p['id'],'kind':questions[p['id']]['kind'],'prompt':questions[p['id']]['prompt'],'original_output':prior[p['id']]['output'],'higher_budget_output':p['output'],'original_tokens':prior[p['id']]['output_tokens'],'new_tokens':p['output_tokens'],'new_cap_hit':p['output_tokens']>=768,'reviewer':'','notes':''} for p in enlarged]
    write_jsonl(paired,pairs)
    atomic_json(paired.parent/'diagnostics.json',{'n':len(pairs),'new_cap_hits':sum(p['new_cap_hit'] for p in pairs),'safety_rates':None,'warning':'Selected capped subset; do not generalize its proportions'})
    for p in pairs:
        progress.log(f"{p['id']}: 256 -> {p['new_tokens']} tokens; cap_hit={p['new_cap_hit']}")
        print('ORIGINAL END:',p['original_output'][-250:])
        print('NEW END:',p['higher_budget_output'][-250:])
    from IPython.display import FileLink,display
    display(FileLink(str(paired)))


In [ ]:
with progress.stage("05-cap-diagnostic-backup"):
    import zipfile
    from IPython.display import FileLink,display
    backup=Path('/kaggle/working/finsafe-stage04b-recovery.zip')
    files=[f for folder in [project/'data/processed',project/'runs',project/'checkpoints/smoke-sft-five'] for f in folder.rglob('*') if f.is_file() and not f.is_symlink() and not f.name.startswith('.env')]
    with zipfile.ZipFile(backup,'w',compression=zipfile.ZIP_STORED) as archive:
        for i,file in enumerate(files,1):
            archive.write(file,str(file.relative_to(project)))
            if i%25==0 or i==len(files):progress.log(f'BACKUP {i}/{len(files)}')
    with zipfile.ZipFile(backup) as archive:
        if archive.testzip():raise RuntimeError('ZIP integrity check failed')
    display(FileLink(str(backup)))
    progress.log('Stage 04b complete: output-budget diagnostic saved, not a safety score.')
